# ImageNet Object/Background Splitter

This notebook:
- asks for an ImageNet image folder and how many images to process
- loads your SAM2 predictor via `scripts/sam2_utils.py`
- runs **grid prompting** to get a foreground/object mask
- writes two images per input:
  - `object/`: object kept, rest filled (white or black)
  - `background/`: background kept, object filled (white or black)

If your `sam2_utils.py` function signatures differ, you only need to edit the **`segment_image_grid(...)`** cell.

In [26]:
import os
import sys
from pathlib import Path
import random
import numpy as np
from PIL import Image

# -----------------------------
# 1) User inputs
# -----------------------------
IMG_DIR = Path("../benchmarks/vggnet16_benchmark2022/imagenet-sample")
N = 1000
FILL_MODE = "white"
SAMPLE_MODE = "first"

assert IMG_DIR.exists(), f"Image folder does not exist: {IMG_DIR}"
assert FILL_MODE in {"white", "black"}, "FILL_MODE must be 'white' or 'black'"
assert SAMPLE_MODE in {"first", "random"}, "SAMPLE_MODE must be 'first' or 'random'"

# Output
OUT_DIR = IMG_DIR.parent / "segmented_outputs"
OBJ_DIR = OUT_DIR / "object"
BG_DIR  = OUT_DIR / "background"
MASK_DIR = OUT_DIR / "masks"  # optional: saved masks for debugging

for d in [OBJ_DIR, BG_DIR, MASK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Saving results to:", OUT_DIR)

# Collect images
img_files = [p for p in IMG_DIR.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'}]
img_files = sorted(img_files)
if SAMPLE_MODE == "random":
    random.shuffle(img_files)
img_files = img_files[:N]

print(f"Found {len(img_files)} images to process.")

Saving results to: ../benchmarks/vggnet16_benchmark2022/segmented_outputs
Found 1000 images to process.


In [27]:
# -----------------------------
# 2) Import your sam2_utils
# -----------------------------
# Provide your XAIV project root (the folder that contains `scripts/`)
# Example: /Users/.../XAIV

PROJECT_ROOT = Path("..")
assert PROJECT_ROOT.exists(), f"Project root does not exist: {PROJECT_ROOT}"

sys.path.insert(0, str(PROJECT_ROOT))

try:
    from scripts.sam2_utils import load_sam2_predictor, run_segmentation_model
    print("Imported: scripts.sam2_utils")
except Exception as e:
    raise ImportError(
        "Could not import scripts.sam2_utils.\n"
        "Make sure PROJECT_ROOT is correct and contains scripts/sam2_utils.py\n"
        f"Original error: {e}"
    )

Imported: scripts.sam2_utils


In [28]:
# -----------------------------
# 3) SAM2 config + predictor
# -----------------------------
# IMPORTANT:
# - Your earlier error was KeyError: 'model_name'.
# - So we *explicitly* set model_name here.
#
# Set this to the SAM2 model id you use in your project.
# If your sam2_utils expects a different key, change it here.

SEG_CFG = {
    "model_name": "facebook/sam2-hiera-large",  # <-- change if you use a different SAM2 checkpoint id
    "device": "cuda" if os.environ.get("CUDA_VISIBLE_DEVICES") not in (None, "", "-1") else "cpu",
}

print("SEG_CFG =", SEG_CFG)
predictor = load_sam2_predictor(SEG_CFG)
print("Loaded SAM2 predictor.")

SEG_CFG = {'model_name': 'facebook/sam2-hiera-large', 'device': 'cpu'}
[SAM2] Using device: cpu
Loaded SAM2 predictor.


In [29]:
# -----------------------------
# 4) Grid prompting segmentation
# -----------------------------
# This function tries a few common call patterns.
# If your run_segmentation_model signature differs, edit ONLY this cell.

def make_grid_points(W: int, H: int, grid_n: int = 12, margin: int = 10):
    """Return (points, labels) for SAM-style prompts.
    points: (K,2) pixel coords (x,y)
    labels: (K,) all 1 (foreground)
    """
    xs = np.linspace(margin, max(margin, W - 1 - margin), grid_n)
    ys = np.linspace(margin, max(margin, H - 1 - margin), grid_n)
    pts = np.array([(float(x), float(y)) for y in ys for x in xs], dtype=np.float32)
    labs = np.ones((pts.shape[0],), dtype=np.int32)
    return pts, labs

def _to_binary_mask(mask):
    """Coerce mask to bool (H,W)."""
    m = np.array(mask)
    if m.ndim == 3:
        # sometimes returned as (N,H,W) or (H,W,1)
        if m.shape[0] in (1,):
            m = m[0]
        elif m.shape[-1] in (1,):
            m = m[..., 0]
    # threshold if float
    if m.dtype != bool:
        m = m > 0.5
    return m.astype(bool)

def _as_numpy(x):
    import numpy as np
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)

def _extract_masks_and_scores(out):
    """
    Returns: (masks_list, scores_list_or_None)
    masks_list: list of (H,W) arrays/bools
    """
    if isinstance(out, dict):
        # Case A: {"masks": (N,H,W), "scores": (N,)}  (or similar)
        if "masks" in out:
            masks = _as_numpy(out["masks"])
            scores = out.get("scores", out.get("iou_scores", out.get("predicted_iou", None)))
            scores = None if scores is None else _as_numpy(scores)
            # normalize shape
            if masks.ndim == 2:
                masks_list = [masks]
            elif masks.ndim == 3:
                masks_list = [masks[i] for i in range(masks.shape[0])]
            else:
                raise ValueError(f"Unexpected masks shape: {masks.shape}")
            return masks_list, (None if scores is None else list(scores))

        # Case B: {"segments": [{"mask": ... , "score": ...}, ...]}
        if "segments" in out:
            segs = out["segments"]
            masks_list = []
            scores_list = []
            for s in segs:
                if isinstance(s, dict):
                    m = s.get("mask", s.get("segmentation", None))
                    if m is None:
                        continue
                    masks_list.append(_as_numpy(m))
                    sc = s.get("score", s.get("iou", s.get("predicted_iou", None)))
                    scores_list.append(sc if sc is not None else None)
            if len(masks_list) == 0:
                raise ValueError(f"'segments' present but no masks found. Segment keys: {list(segs[0].keys()) if len(segs)>0 else 'EMPTY'}")
            # if all scores None -> return None
            if all(sc is None for sc in scores_list):
                return masks_list, None
            # replace None with very small score
            scores_list = [(-1e9 if sc is None else float(sc)) for sc in scores_list]
            return masks_list, scores_list

        # Case C: {"mask": (H,W)}
        if "mask" in out:
            return [_as_numpy(out["mask"])], None

        raise ValueError(f"Unknown dict output keys from run_segmentation_model: {list(out.keys())}")

    # Non-dict fallback
    if isinstance(out, (tuple, list)):
        return [_as_numpy(out[0])], None
    return [_as_numpy(out)], None

def _to_binary_mask(mask):
    import numpy as np
    m = np.array(mask)
    if m.ndim == 3:
        # (H,W,1) or (1,H,W)
        if m.shape[0] == 1:
            m = m[0]
        elif m.shape[-1] == 1:
            m = m[..., 0]
    if m.dtype != bool:
        m = m > 0.5
    return m.astype(bool)

def _mask_from_any(m):
    """
    Converts various mask formats into a (H,W) numpy array.
    Handles numpy/torch masks and COCO-style RLE dicts.
    """
    import numpy as np

    # torch -> numpy
    try:
        import torch
        if isinstance(m, torch.Tensor):
            m = m.detach().cpu().numpy()
    except Exception:
        pass

    if isinstance(m, np.ndarray):
        return m

    if isinstance(m, dict):
        # COCO RLE: {"size":[H,W], "counts": ...}
        if "size" in m and "counts" in m:
            # Best option: pycocotools
            try:
                from pycocotools import mask as mask_utils
                decoded = mask_utils.decode(m)
                if decoded.ndim == 3:
                    decoded = decoded[..., 0]
                return decoded
            except Exception:
                # Fallback for uncompressed RLE where counts is a list
                counts = m["counts"]
                H, W = m["size"]
                if isinstance(counts, list):
                    flat = np.zeros(H * W, dtype=np.uint8)
                    idx = 0
                    val = 0
                    for run in counts:
                        if run > 0:
                            flat[idx:idx+run] = val
                        idx += run
                        val = 1 - val
                    # COCO uses column-major order
                    return flat.reshape((W, H)).T
                raise ValueError(
                    "Mask is RLE dict but couldn't decode. "
                    "Install pycocotools: pip install pycocotools"
                )

        # if your mask dict is different, show keys
        raise ValueError(f"Unrecognized mask dict keys: {list(m.keys())}")

    return np.asarray(m)

def segment_image_grid(pil_img: Image.Image, grid_n: int = 12):
    """Returns a binary object mask (H,W) using grid prompts, wired to your sam2_utils.run_segmentation_model."""
    # PIL -> numpy float32 in [0,1]
    image_np = np.asarray(pil_img.convert("RGB"), dtype=np.float32) / 255.0
    H, W = image_np.shape[:2]

    # grid prompts (x,y) + labels (all foreground)
    points, labels = make_grid_points(W, H, grid_n=grid_n)

    out = run_segmentation_model(
        predictor,
        image_np,
        grid_size=grid_n,        # harmless even if points are provided
        point_coords=points,
        point_labels=labels,
    )

    # out is: list of dicts: [{"bbox":..., "mask":..., "score":...}, ...]
    assert isinstance(out, list) and len(out) > 0, f"Unexpected output type/empty: {type(out)}"

    # pick best segment (highest score)
    best = max(out, key=lambda d: float(d.get("score", -1e9)))
    raw_mask = best["mask"]

    # decode mask -> (H,W) numpy
    mask_np = _mask_from_any(raw_mask)   # <-- use the helper below
    mask = _to_binary_mask(mask_np)

    print("picked score:", float(best.get("score", -1)), "mask shape:", mask.shape, "mask mean:", float(mask.mean()))
    return mask

In [30]:
# -----------------------------
# 5) Run + write object/background images
# -----------------------------
from torchvision.transforms import functional as F
from torchvision.transforms import InterpolationMode

FILL = 255 if FILL_MODE == "white" else 0
SIZE = 224

for idx, p in enumerate(img_files):
    img0 = Image.open(p).convert("RGB")

    # VGG-friendly spatial preprocessing (no normalization for saving images)
    img = F.resize(img0, SIZE, interpolation=InterpolationMode.BILINEAR, antialias=True)
    img = F.center_crop(img, [SIZE, SIZE])

    arr = np.array(img)  # (224,224,3) uint8

    # IMPORTANT: segment on the SAME preprocessed image so mask matches
    mask = segment_image_grid(img, grid_n=6)



    if mask.shape[:2] != arr.shape[:2]:
        raise ValueError(f"Mask shape {mask.shape} does not match image shape {arr.shape}")

    fill_bg = np.ones_like(arr, dtype=np.uint8) * FILL

    obj_img = np.where(mask[..., None], arr, fill_bg)
    bg_img  = np.where(~mask[..., None], arr, fill_bg)

    Image.fromarray(obj_img).save(OBJ_DIR / p.name)
    Image.fromarray(bg_img).save(BG_DIR / p.name)

    # Optional: save mask for debugging
    m = (mask.astype(np.uint8) * 255)
    Image.fromarray(m).save(MASK_DIR / (p.stem + "_mask.png"))

print("Done! Outputs:")
print(" -", OBJ_DIR)
print(" -", BG_DIR)
print(" -", MASK_DIR)

[SAM2] Found 3 segments after filtering
picked score: 0.9458848237991333 mask shape: (224, 224) mask mean: 0.9673349808673469
[SAM2] Found 3 segments after filtering
picked score: 0.8262248635292053 mask shape: (224, 224) mask mean: 0.8061623086734694
[SAM2] Found 3 segments after filtering
picked score: 0.9320828914642334 mask shape: (224, 224) mask mean: 0.9130062181122449
[SAM2] Found 3 segments after filtering
picked score: 0.96719890832901 mask shape: (224, 224) mask mean: 0.9724968112244898
[SAM2] Found 3 segments after filtering
picked score: 0.8890488743782043 mask shape: (224, 224) mask mean: 0.6780731823979592
[SAM2] Found 3 segments after filtering
picked score: 0.9267801642417908 mask shape: (224, 224) mask mean: 0.960578762755102
[SAM2] Found 3 segments after filtering
picked score: 0.953296959400177 mask shape: (224, 224) mask mean: 0.9948580994897959
[SAM2] Found 3 segments after filtering
picked score: 0.9270256757736206 mask shape: (224, 224) mask mean: 0.4903140943877